# Customer Churn — Responsible Baseline Notebook
This notebook is an exploratory, reproducible audit of the supplied 12-row CSV. It is **not** evidence of production readiness. The data owner must confirm provenance, consent, churn definition, prediction timing and representativeness before modelling.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Place customer-churn-training.csv in the same folder as this notebook.
DATA_PATH = Path('customer-churn-training.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Schema and quality checks
Check dimensions, types, missingness, duplicate rows, unique identifiers, value ranges and target balance. A zero count in this tiny file does not prove the wider data pipeline is clean.

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isna().sum())
print('\nExact duplicate rows:', df.duplicated().sum())
print('\nUnique customer IDs:', df['customer_id'].nunique())
print('\nTarget counts:\n', df['churned'].value_counts(dropna=False))
print('\nTarget proportions:\n', df['churned'].value_counts(normalize=True, dropna=False))
print('\nNumeric summary:\n', df.describe())
print('\nPlan type by target:\n', pd.crosstab(df['plan_type'], df['churned']))

## 2. Feature review and leakage controls
`customer_id` is an identifier and must not be a predictor. `churned` is the label, never a feature. Confirm that all other variables are measured before the prediction snapshot. `plan_type` perfectly separates the label in this sample, so treat it as a leakage/sampling-artifact investigation and exclude it from initial modelling until explained.

In [ ]:
feature_columns = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
X = df[feature_columns].copy()
y = df['churned'].astype(int)
assert set(y.unique()).issubset({0, 1})
assert 'customer_id' not in X.columns
assert 'churned' not in X.columns
print('Diagnostic feature set:', feature_columns)
print('Plan-type/target crosstab (investigate):\n', pd.crosstab(df['plan_type'], y))

## 3. Non-ML baseline: majority class
The majority class in this file is not churned (0). Predicting 0 for every row is a transparent reference baseline. Accuracy is 58.3%, but churn recall is 0%; therefore accuracy alone is not a sufficient metric. This is a descriptive score on the supplied records, not a test-set estimate.

In [ ]:
majority = int(y.mode().iloc[0])
baseline_pred = np.repeat(majority, len(y))
print('Majority label:', majority)
print('Accuracy (descriptive, same data):', accuracy_score(y, baseline_pred))
print('Confusion matrix [ [TN, FP], [FN, TP] ]:\n', confusion_matrix(y, baseline_pred, labels=[0, 1]))
print(classification_report(y, baseline_pred, labels=[0, 1], zero_division=0))

## 4. Optional transparent triage rule (not a trained model)
The following example flags a record for human review when inactivity is at least 14 days OR support tickets are at least 4. These thresholds are illustrative only and were not approved by a business owner. The apparent in-sample results must not be presented as generalisation performance or used to contact customers without approval.

In [ ]:
triage_pred = ((df['last_login_days'] >= 14) | (df['support_tickets'] >= 4)).astype(int)
print('Descriptive confusion matrix:\n', confusion_matrix(y, triage_pred, labels=[0, 1]))
print(classification_report(y, triage_pred, labels=[0, 1], zero_division=0))
print('Flagged for manual review:', int(triage_pred.sum()), 'of', len(df))

## 5. Why no model score is reported
There are only 12 records, no timestamps, no documented sampling frame and no train/test indicator. A random split would be unstable and could not test future performance. The perfect plan-type separation is unexplained. Do not fit/tune a model and report a score as if it were validated. Obtain a larger time-stamped dataset and use customer-grouped temporal train/validation/test splits.

## 6. Evaluation plan before any future training
Pre-register: confusion matrix; precision; recall; F1; specificity; PR-AUC; ROC-AUC as secondary; calibration plot and Brier score; threshold-specific expected cost; subgroup error-rate analysis where lawful and sufficiently powered; and qualitative review of false positives/negatives. Tune thresholds only on validation data and report final metrics once on an untouched future test period.

## 7. Responsible-use decision
**Decision: no production ML.** Use a transparent human-reviewed service process while provenance, permission, label definition, temporal ordering, sample representativeness and the plan-type anomaly are resolved. No automatic cancellation, price change, service restriction or adverse action is permitted.